# Financial Risk Monitoring Pipeline

This notebook presents the Python/data-preparation layer of a portfolio case study that combines transaction monitoring, complaint-theme analysis, PostgreSQL scenario logic and Tableau reporting.

The public portfolio version uses repository-local files only. The original coursework used MongoDB Atlas for document ingestion; credentials and external database dependencies have been removed here for reproducibility.


## Data sources

- **IBM AMLSim example dataset** — account and transaction snapshots used for monitoring-rule demonstrations.
- **Banking consumer complaint analysis dataset** — complaint records used for text processing and theme analysis.

The repository contains the exact public/sample snapshots used in the portfolio refactor. No employer, client or confidential data are included.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
DATA = ROOT / "data"

accounts = pd.read_csv(DATA / "raw" / "accounts.csv")
transactions = pd.read_csv(DATA / "raw" / "transactions.csv")
complaints = pd.read_json(DATA / "raw" / "complaints.json")

print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)
print("Complaints:", complaints.shape)


## 1. Data quality review

The source complaint data contain missing `Sub-issue` and `State` values. The portfolio version does **not** invent replacement geography. Missing values remain explicit unless a deterministic transformation is available.


In [ ]:
quality = pd.DataFrame({
    "rows": [len(accounts), len(transactions), len(complaints)],
    "missing_values": [
        int(accounts.isna().sum().sum()),
        int(transactions.isna().sum().sum()),
        int(complaints.isna().sum().sum()),
    ],
}, index=["accounts", "transactions", "complaints"])
quality


## 2. Complaint clustering output

The original analytical workflow combined `Issue` and `Sub-issue`, cleaned the text, generated sentence embeddings, reduced dimensionality and applied K-Means/Agglomerative clustering.

To keep the public repository lightweight and stable, the precomputed clustering output is stored in `data/processed/complaints_clustered.csv`. This avoids requiring a large language-embedding model download every time the notebook is opened.


In [ ]:
clustered = pd.read_csv(DATA / "processed" / "complaints_clustered.csv")

cluster_summary = (
    clustered.groupby(["final_cluster", "complaint_theme"])
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

cluster_summary.head(17)


In [ ]:
theme_counts = clustered["complaint_theme"].value_counts().sort_values()

plt.figure(figsize=(9, 5))
theme_counts.plot(kind="barh")
plt.title("Complaint themes after cluster consolidation")
plt.xlabel("Records")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 3. Scenario-based transaction monitoring

The linked SQL/Tableau project uses six core monitoring categories:

1. transaction greater than 10,000;
2. high total outflow by sender;
3. two-way circular transaction pattern;
4. source fraud indicator;
5. transaction greater than 1,000,000;
6. transaction equal to the sender's initial balance.

The SQL implementation is in `sql/03_monitoring_scenarios.sql`. The processed alert-event snapshot used for the dashboard is available below.


In [ ]:
alerts = pd.read_csv(DATA / "processed" / "transaction_scenarios.csv")

scenario_counts = alerts["scenario_name"].value_counts()
scenario_counts


In [ ]:
plt.figure(figsize=(9, 5))
scenario_counts.sort_values().plot(kind="barh")
plt.title("Alert observations by monitoring scenario")
plt.xlabel("Alert observations")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Interpretation note

The dashboard uses **alert observations**, not unique investigations. For the circular-pattern rule, both transaction legs are represented and the underlying matching logic is directional. This is retained so that the processed data remain consistent with the published Tableau story.


## 4. Dashboard-ready client risk

The visualisation project grouped clients using the combined number of complaint and alert observations:

- Low: 0–1
- Medium: 2–4
- High: 5–7
- Prohibited / very high: more than 7

These bands are illustrative analytical categories rather than production risk policy.


In [ ]:
client_risk = pd.read_csv(DATA / "processed" / "client_risk.csv")

client_risk["risk_band"].value_counts()


## 5. Portfolio outcome

This case study demonstrates an end-to-end flow from heterogeneous data to stakeholder reporting:

`public/sample data → preparation → complaint themes → relational model → SQL monitoring rules → dashboard-ready outputs → Tableau`

The interactive Tableau workbook and dashboard screenshot are available in the `dashboard/` folder.
